#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

#Read from Bronze Table

In [0]:
df = spark.table('workspace.bronze.erp_cust_az12')

#Data Transformations

On my initial exploration with only 10 rows I thouggh we only needed trimming and renamieng the columns, after checking the repo of Baara I discover that actually if we went more in depth into the data, the CID had different sizes where some had an extra NAS at the beggining while the rest didnt so in order to clean up the customer id we remove the NAS part on the rows which had this, regarding the birthdate eventhough the format was ok the problem was that there were date that were in the future (impossible values), so we did a birthdate validation where we comare the value in the row with the current day and if the value was greater than the current date then we will put a NONE/NULL value instead. Regarding the column gender at first it seems as it the values were already normalize but if we went depper there were not only male, or female values but F, M, Null values, so we had to normalize the values in the column where Male,m became Male and Female,F became Female and any value which is not in this categories (NULL, "", Uknown, etc) became n/a. Normalizing the values in the column to (Male, Female or n/a), finally we did the renaming columns we initially identify and do sanity checks to see the final dataframe beofre writing it to a table.

##Trimming

In [0]:

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Customer ID Cleanup

In [0]:
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)

##Birthdate Validation

In [0]:
df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(), None)
     .otherwise(col("bdate"))
)

##Gender Normalization

In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)

## Renaming columns

In [0]:
RENAME_MAP = {
    "CID": "customer_id",
    "BDATE": "birth_date",
    "GEN": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity checks Dataframe

In [0]:
df.limit(10).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")

##Sanity checks Silver Table

###Customer ID is unique

In [0]:
%sql
SELECT
  *
FROM (
  SELECT
    customer_id,
    COUNT(*) OVER(PARTITION BY customer_id) CheckPK
  FROM silver.erp_customers
)subquery
WHERE CheckPK > 1

In [0]:
%sql
SELECT * FROM silver.erp_customers limit 10